# Goal 1: from protein sequences to an alignment network

## Purpose

This is the smallest executable version of the project. It shows what every
later experiment does: accept named amino-acid sequences, score every unique
pair, and interpret sufficiently strong pairwise relationships as edges in an
undirected graph. It uses deliberately tiny example sequences so each object
can be inspected directly; it is a software demonstration, not a biological
experiment.

The project does **not** implement its own Smith--Waterman algorithm. Exact
local alignment is delegated to Biopython's maintained `PairwiseAligner`
([Cock et al., 2009](https://doi.org/10.1093/bioinformatics/btp163)), following
the local-alignment principle introduced by
[Smith and Waterman (1981)](https://doi.org/10.1016/0022-2836(81)90087-5).
Reusable project code only validates inputs, coordinates all-versus-all
scoring, and constructs graphs.

```text
named sequences
    -> configured local aligner
    -> symmetric all-pairs score matrix
    -> explicit edge rule
    -> NetworkX graph
```

The reusable implementations called below are:

- `src/protein_alignment_networks/pipeline.py`: all-pairs Biopython scoring;
- `src/protein_alignment_networks/graphs.py`: score-to-graph conversion;
- `src/protein_alignment_networks/io.py`: sequence validation and FASTA I/O.

Notebook 02 applies the complete pipeline to a real Pfam pilot. Notebook 03 is
the main experiment and studies how input collections and edge rules change
the resulting graphs.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from Bio import Align
from Bio.Align import substitution_matrices

from protein_alignment_networks import (
    pairwise_score_matrix,
    score_matrix_to_graph,
)

## 1. Configure exact local protein alignment

`mode="local"` asks for the best matching subsequences rather than forcing the
sequences to align end to end. BLOSUM62 supplies empirically derived amino-acid
substitution scores ([Henikoff and Henikoff, 1992](https://doi.org/10.1073/pnas.89.22.10915)).
Opening a gap costs more than extending it, an affine-gap model efficiently
implemented by the recurrence of
[Gotoh (1982)](https://doi.org/10.1016/0022-2836(82)90398-9).

The BLOSUM62, gap-open, and gap-extension settings are conventional protein
alignment defaults, not a universal biological optimum. They are kept visible
because a score has meaning only together with its scoring configuration.


In [2]:
aligner = Align.PairwiseAligner(
    mode='local',
    substitution_matrix=substitution_matrices.load('BLOSUM62'),
    open_gap_score=-11,
    extend_gap_score=-1,
)
print(aligner.algorithm)

Gotoh local alignment algorithm


## 2. Inspect one alignment

For one pair, `score()` returns the optimum local score and `align()` returns
the corresponding aligned regions. The code output below is the result itself;
no numerical result is copied into the surrounding prose.


In [3]:
sequence_a = 'PAWHEAE'
sequence_b = 'HEAGAWGHEE'
alignments = aligner.align(sequence_a, sequence_b)
print(f'Score: {alignments.score:g}')
print(alignments[0] if len(alignments) else 'No positive local alignment')

Score: 17
target            1 AW-HE 5
                  0 ||-|| 5
query             4 AWGHE 9



## 3. Compute every unique pair

For $n$ sequences there are $n(n-1)/2$ non-self pairs. The reusable
`pairwise_score_matrix()` function calls the configured Biopython aligner once
per unique pair, mirrors the result across the diagonal, and returns a labelled
matrix. Later scripts also convert this matrix to a canonical table with one
row per unordered pair, which is easier to merge with BLAST and DEDAL outputs.


In [4]:
sequences = {
    'protein_a': 'PAWHEAE',
    'protein_b': 'HEAGAWGHEE',
    'protein_c': 'MKTAYIAKQRQISFVKSHFSRQ',
    'protein_d': 'PAWHDQE',
}
scores = pairwise_score_matrix(sequences, aligner=aligner)
scores

protein_id,protein_a,protein_b,protein_c,protein_d
protein_id,,,,
protein_a,44.0,17.0,8.0,36.0
protein_b,17.0,62.0,8.0,19.0
protein_c,8.0,8.0,109.0,8.0
protein_d,36.0,19.0,8.0,46.0


## 4. Convert scores into a graph

Every sequence becomes a node, including sequences with no retained edge. In
this toy example an absolute threshold retains a pair when its score is at
least the configured value. The threshold demonstrates the programming step
only: raw alignment scores depend on sequence length, composition, scoring
parameters, and the selected collection.

Notebook 03 therefore compares several transparent edge rules rather than
claiming one absolute cutoff is correct. The displayed node and edge counts are
computed directly from the graph object below.


In [5]:
demonstration_threshold = 20.0
graph = score_matrix_to_graph(scores, threshold=demonstration_threshold)
print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')
list(graph.edges(data=True))

Nodes: 4
Edges: 1


[('protein_a', 'protein_d', {'score': 36.0})]

## What this notebook establishes

The complete minimal data flow works in memory, and the reusable code—not the
notebook—contains the tested implementations. What this notebook does **not**
establish is whether the demonstration threshold has biological meaning or
whether results generalise beyond the toy sequences. Those questions require
real data, separate method outputs, explicit provenance, and threshold
sensitivity; Notebook 02 adds the first three and Notebook 03 performs the
large-scale analysis.

## References

Full BibTeX records are stored in `references/references.bib`.

- Smith, T. F. and Waterman, M. S. (1981), *Identification of Common Molecular
  Subsequences*. `smith1981identification`.
- Gotoh, O. (1982), *An Improved Algorithm for Matching Biological
  Sequences*. `gotoh1982improved`.
- Henikoff, S. and Henikoff, J. G. (1992), *Amino Acid Substitution Matrices
  from Protein Blocks*. `henikoff1992amino`.
- Cock, P. J. A. et al. (2009), *Biopython: Freely Available Python Tools for
  Computational Molecular Biology and Bioinformatics*. `cock2009biopython`.
